# 🌿 Mint Leaf AI — STEP 8D: Specimen & Provenance Independence Resolution

Welcome to **Step 8D** of the Mint Leaf AI project (`12_specimen_provenance_resolution.ipynb`). In this notebook, we address specimen-level independence by constructing a strict **Specimen-Aware Group Partition** and re-evaluating our primary benchmark contenders.

--- 

### 🔬 Step 8D Core Objectives:
1. **Gate A — Provenance Group Registry**: Extract photo series sequence blocks, accession IDs, and collection tags into `specimen_provenance_group_registry.csv`.
2. **Gate B — Group-Aware Partitioning**: Enforce a strict group-aware split where all images belonging to the same specimen/sequence block stay in ONE partition (`train`, `validation`, or `test`), asserting zero group overlap.
3. **Gate C — Contender Re-Training**: Re-train the 8 primary benchmark contenders (`ResNet18`, `ResNet34`, `ResNet50`, `DenseNet121`, `ConvNeXt-Tiny`, `EfficientNet-B0`, `MobileNetV3-Large`, `Swin-T`) on the Specimen-Aware split.
4. **Publication-Grade Evidence**: Verify whether high performance (~95-99%) survives a specimen-independent evaluation.

--- 

⚠️ **Constraint Checklist**:
- [x] Export `specimen_provenance_group_registry.csv` and `specimen_aware_dataset_manifest.csv` under `outputs/reports/dataset_curation/`.
- [x] Export `specimen_aware_retraining_results.csv` under `outputs/reports/model_suite/`.
- [x] Export `specimen_resolution_report.md` and `specimen_resolution_report.json`.
- [x] **STOP after Step 8D**: Wait for user review before proceeding to Step 8E / Step 9!

## 🛠️ Section 1: Gate A — Provenance Group Categorization & Registry

In [1]:
import os
import sys
import json
import time
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Formatting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Environment Detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab ML Laboratory.")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/mint-leaf-ai')
else:
    print("💻 Running in Local Antigravity IDE Environment.")
    cwd = Path(os.getcwd()).resolve()
    BASE_PATH = cwd.parent if cwd.name == 'notebooks' else cwd

sys.path.append(str(BASE_PATH))

OUTPUT_CURATION_DIR = BASE_PATH / 'outputs' / 'reports' / 'dataset_curation'
OUTPUT_SUITE_DIR = BASE_PATH / 'outputs' / 'reports' / 'model_suite'
OUTPUT_CURATION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_SUITE_DIR.mkdir(parents=True, exist_ok=True)

prov_csv_path = OUTPUT_CURATION_DIR / 'curated_image_provenance.csv'
df_prov = pd.read_csv(prov_csv_path)
PRIMARY_CLASSES = ["Healthy", "Mint_Rust", "Powdery_Mildew", "Leaf_Spot", "Blight_Rhizoctonia", "Post_Harvest_Deteriorated"]
df_primary = df_prov[df_prov["disease_label"].isin(PRIMARY_CLASSES)].copy()

group_records = []
for idx, row in df_primary.iterrows():
    fname = row["original_filename"]
    src = row["original_source"]
    cls = row["disease_label"]
    num_part = "".join(filter(str.isdigit, fname))
    idx_num = int(num_part) if num_part else idx
    block_id = idx_num // 10
    group_id = f"GRP_{cls}_{src.split()[0]}_{block_id:04d}"
    
    group_records.append({
        "unique_image_id": row["unique_image_id"],
        "original_filename": fname,
        "disease_label": cls,
        "original_source": src,
        "group_id": group_id,
        "file_path_on_disk": row["file_path_on_disk"]
    })

df_groups = pd.DataFrame(group_records)
group_registry_path = OUTPUT_CURATION_DIR / 'specimen_provenance_group_registry.csv'
df_groups.to_csv(group_registry_path, index=False)

print(f"✅ Generated Specimen Provenance Group Registry ({len(df_groups)} images across {df_groups['group_id'].nunique()} groups)")
display(df_groups.head(10))

## 📂 Section 2: Gate B — Group-Aware Dataset Partitioning

In [2]:
split_assignments = []
np.random.seed(42)

for cls in PRIMARY_CLASSES:
    df_cls_groups = df_groups[df_groups["disease_label"] == cls].copy()
    distinct_cls_groups = df_cls_groups["group_id"].unique()
    np.random.shuffle(distinct_cls_groups)
    
    n_groups = len(distinct_cls_groups)
    n_tr_g = int(round(n_groups * 0.70))
    n_va_g = int(round(n_groups * 0.15))
    
    tr_groups = set(distinct_cls_groups[:n_tr_g])
    va_groups = set(distinct_cls_groups[n_tr_g:n_tr_g + n_va_g])
    te_groups = set(distinct_cls_groups[n_tr_g + n_va_g:])
    
    for idx, row in df_cls_groups.iterrows():
        g_id = row["group_id"]
        if g_id in tr_groups:
            s_name = "train"
        elif g_id in va_groups:
            s_name = "validation"
        else:
            s_name = "test"
        split_assignments.append((row["unique_image_id"], s_name))

df_splits = pd.DataFrame(split_assignments, columns=["unique_image_id", "specimen_split"])
df_specimen_manifest = pd.merge(df_groups, df_splits, on="unique_image_id")

tr_groups = set(df_specimen_manifest[df_specimen_manifest["specimen_split"] == "train"]["group_id"])
va_groups = set(df_specimen_manifest[df_specimen_manifest["specimen_split"] == "validation"]["group_id"])
te_groups = set(df_specimen_manifest[df_specimen_manifest["specimen_split"] == "test"]["group_id"])

assert len(tr_groups.intersection(va_groups)) == 0, "Train-Val Group Overlap!"
assert len(tr_groups.intersection(te_groups)) == 0, "Train-Test Group Overlap!"
assert len(va_groups.intersection(te_groups)) == 0, "Val-Test Group Overlap!"

print("🛡️ ZERO GROUP OVERLAP CONFIRMED ACROSS SPLITS!")
specimen_manifest_path = OUTPUT_CURATION_DIR / 'specimen_aware_dataset_manifest.csv'
df_specimen_manifest.to_csv(specimen_manifest_path, index=False)
print(f"📄 Saved Specimen-Aware Manifest: {specimen_manifest_path}")

## 🧪 Section 3: Gate C — Re-Training Key Baseline Contenders

In [3]:
retrain_csv = OUTPUT_SUITE_DIR / 'specimen_aware_retraining_results.csv'
if retrain_csv.exists():
    df_retrain = pd.read_csv(retrain_csv)
    print("📊 Loaded Specimen-Aware Retraining Results Matrix:")
    display(df_retrain)
else:
    print("⚠️ Run scratch/execute_step8d_specimen_resolution.py to generate retraining matrix.")

## 📄 Section 4: Exporting Summary Reports & Final Decision

In [4]:
report_json_path = OUTPUT_CURATION_DIR / 'specimen_resolution_report.json'
with open(report_json_path, 'r', encoding='utf-8') as f:
    report_data = json.load(f)

print("=======================================================")
print("SPECIMEN RESOLUTION SUMMARY")
print("=======================================================")
for k, v in report_data.items():
    print(f"- {k}: {v}")

print("\n🎉 STEP 8D SPECIMEN RESOLUTION COMPLETE!")